# Acts 1–2 — Agents, harnesses, and context

These are the lesson cells tested in Colab. Run this notebook from top to bottom; it also works independently of the other act notebooks.

**Setup:** use a Python 3.13+ kernel. The next cells find or clone the repository and install its pinned direct dependencies. If your Colab setup already did this, skip those two cells and run the imports and settings below from the repository directory.

All imports are grouped here, above the lessons. Set any model environment override before running the lesson imports.


In [ ]:
from pathlib import Path
from getpass import getpass
import tomllib
import os
import subprocess
import sys

root = Path.cwd()
if not (root / "formaggio" / "config.py").exists():
    if (root.parent / "formaggio" / "config.py").exists():
        root = root.parent
    else:
        root = root / "agent-harness"
        if not root.exists():
            subprocess.run(["git", "clone", "https://github.com/dlops-io/agent-harness.git", str(root)], check=True)
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
print("Repository:", root)

In [ ]:
if sys.version_info < (3, 13):
    raise RuntimeError("Select a Python 3.13+ notebook kernel before continuing.")
project = tomllib.loads(Path("pyproject.toml").read_text())
subprocess.run([sys.executable, "-m", "pip", "install", *project["project"]["dependencies"]], check=True)
subprocess.run([sys.executable, "-m", "scripts.check_sdk"], check=True)

In [ ]:
import os
from getpass import getpass
from pathlib import Path

from agent_framework import Agent
from agent_framework.openai import OpenAIChatClient
from openai import AsyncOpenAI

from acts.act1_agent import build_agent
from acts.act2_context import build_act2, preview_context
from acts.act3_workflow import build_act3
from acts.act4_harness import build_act4
from acts.act5_skills import build_act5
from acts.act6_composition import build_act6
from cli import (
    print_agent_result,
    print_cart,
    print_workflow_result,
    print_harness_result,
    print_composition_result,
)
from formaggio.config import MODEL
from formaggio.agents.context import (
    customer_ask, customer_message, instructions, load_scenario,
)
from formaggio.agents.harness import Harness
from formaggio.agents.layers import Trace, Budget, Context, CartCheck
from formaggio.agents.runtime import FUNCTION_LIMITS, MODEL_OPTIONS
from formaggio.agents.tools import build_tools
from formaggio.operations.observability import Recorder
from formaggio.shop.data_models import AgentReply
from formaggio.shop.store import Store
from formaggio.operations.chat_view import show_chat


## Before we begin

We will build a cheese-shop assistant in six steps. The shop data and rules are fictional classroom examples.

The cells use live model calls by default. Set `USE_FIXTURES = True` to use scripted responses in Acts 3–6. Acts 1–2 still use the live model.

When a review is needed, you will see the cart or email draft and type `approve` or `decline`. Orders are simulated, and approved emails are saved as local HTML files; no purchases or emails are sent.


In [ ]:
SCENARIO = "standard"  # Keep this the same for Acts 1–2.
USE_FIXTURES = False

DB_PATH = Path("outputs/colab.sqlite")
OUTPUT_ROOT = Path("outputs/colab_artifacts")
MEMORY_PATH = Path("outputs/colab_memory.sqlite")
DB_PATH.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


def ask_for_decision():
    while True:
        decision = input("Type approve or decline: ").strip().lower()
        if decision in {"approve", "decline"}:
            return decision
        print("Please enter approve or decline.")


async def review_order(ticket):
    print("\n👤 Manager review")
    print(customer_ask(ticket.request))
    print_cart(ticket.report)
    return ask_for_decision()


async def review_email(review):
    print("\n👤 Review this email draft")
    print(f"To: {review.draft.recipient}")
    print(f"Subject: {review.draft.subject}\n")
    print(review.draft.body)
    return ask_for_decision()


print(f"Model: {MODEL}")
print("Acts 3–6:", "scripted responses" if USE_FIXTURES else "live model")


### API key

If you already configured `OPENAI_API_KEY` in Colab, this cell reuses it. Otherwise it asks privately when live calls are enabled. The key is not printed or saved in the notebook.


In [ ]:
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")


## Act 1 — Build an agent

An agent combines a model, instructions, and tools. Here, the model can look up cheeses, check stock, calculate prices, and propose a tasting.

The SDK handles the loop: call the model, run its requested tools, and return the results until it gives a final answer.

First, run the agent directly. It can propose a cart, but it cannot place an order.


In [ ]:
store = Store()
request = load_scenario(SCENARIO)
print("👤 Customer ask:\n" + customer_ask(request))

async with AsyncOpenAI(timeout=45, max_retries=0) as api:
    client = OpenAIChatClient(
        model=MODEL,
        async_client=api,
        function_invocation_configuration=FUNCTION_LIMITS,
    )
    agent = Agent(
        client=client,
        name="FormaggioAssistant",
        instructions=instructions(store),
        tools=build_tools(store, request, None, None),
        default_options={**MODEL_OPTIONS, "response_format": AgentReply},
    )
    bare_response = await agent.run(customer_message(request))

print("\n💬 Agent's response:\n" + bare_response.value.message)


### Add checks around the agent

The harness runs the same agent definition with four named layers:

- **Trace:** show and record what happened.
- **Budget:** limit model calls, tool calls, and runtime.
- **Context:** choose the extra information supplied to the agent.
- **CartCheck:** independently check the final proposed cart.

`build_agent` contains the agent definition shown above, with hooks for these layers. Calling `.run(...)` starts a fresh run using the chosen scenario.

The cart check reports problems; it does not repair the cart or place an order.


In [ ]:
act1 = (
    Harness(build_agent, model=MODEL, scenario=SCENARIO)
    .add(Trace())
    .add(Budget(model_calls=8, tool_calls=20, seconds=120))
    .add(Context("basic"))
    .add(CartCheck())
)

with Recorder(DB_PATH) as recorder:
    act1_result = await act1.run(recorder, progress=print)

print_agent_result(act1_result)


### View the recorded conversation and checks

The left panel shows the recorded task, the final assistant answer when available, and application results. The right panel shows checks, review decisions, and expandable evidence. Internal model replies remain in the evidence panel.

This cell reads the saved run; it does not call the model or repeat any action. Using `DB_PATH` works after the run cell closes its recorder. Add `backstage=False` to show only the request, response, and application results.


In [ ]:
show_chat(DB_PATH, act1_result["run_id"])


## Act 2 — Give the agent useful context

Context engineering means deciding what information the model should see for this task.

We will compare the same agent with basic and enriched context. The enriched version includes relevant customer preferences and a starting list of cheeses filtered using stock, destination, and allergy information.

We keep the model, instructions, tools, and customer request the same. First inspect the added information, then compare the two runs.

Filtering the starting list does not replace checking the final cart. Also, one successful run is not enough to prove that one approach is better.


In [ ]:
context_preview = preview_context(SCENARIO)

print("📚 Information added in enriched mode:")
for source in context_preview["enriched"]["sources"]:
    print(f"- {source['source']}: {source['reason']}")

print("\n🔎 Cheeses excluded from the starting list:")
for product in context_preview["enriched"]["excluded_products"]:
    print(f"- {product['product_id']}: {', '.join(product['reasons'])}")

context_results = {}
for mode in ("basic", "enriched"):
    print(f"\n🧪 Running {mode} context")
    lesson = build_act2(scenario=SCENARIO, mode=mode, model=MODEL)
    with Recorder(DB_PATH) as recorder:
        context_results[mode] = await lesson.run(recorder, progress=print)
    print_agent_result(context_results[mode])

print("\n📊 Comparison")
for mode, result in context_results.items():
    print(
        f"{mode}: cart check = {result['cart_check_status']}, "
        f"model calls = {result['model_calls']}, "
        f"tool calls = {result['tool_calls']}"
    )


### View the recorded conversation and checks

The left panel shows the recorded task, the final assistant answer when available, and application results. The right panel shows checks, review decisions, and expandable evidence. Internal model replies remain in the evidence panel.

This cell reads the saved run; it does not call the model or repeat any action. Using `DB_PATH` works after the run cell closes its recorder. Add `backstage=False` to show only the request, response, and application results.


In [ ]:
for mode, result in context_results.items():
    show_chat(DB_PATH, result["run_id"], title=f"Act 2 · {mode} context")


### Check the lesson automatically

From the repository terminal, run `python -m tests -p test_notebook.py` to execute the saved lesson cells with local scripted responses. The tests skip cloning, package installation, and credential entry; they use temporary output files and simulate review decisions. Run `python -m tests` for the full offline suite.
